# Top services, per group
Stability selection using bootstrap.

In [ ]:
import pandas as pd
import numpy as np
import ast
from collections import Counter

# 1. Specify your dataset filename here
FILE_PATH = 'classified_architectures_with_metadata.csv'

try:
    df = pd.read_csv(FILE_PATH)
except FileNotFoundError:
    print(f"Error: File '{FILE_PATH}' not found. Please update the FILE_PATH variable with your actual filename.")
    exit()

# Define the groups and their corresponding binary indicator columns from your file
groups = {
    'hpc': 'belongs_hpc',
    'edge': 'belongs_edge',
    'serverless': 'belongs_serverless',
    'faas': 'belongs_serverless_faas',
    'strictly_edge': 'belongs_edge_no_cloudfront'
}

# Robust helper function to parse the 'services' column
def parse_services(val):
    if pd.isna(val):
        return []
    val_str = str(val).strip()
    # Check if it is formatted as a Python list string e.g., ['S3', 'EC2']
    if val_str.startswith('[') and val_str.endswith(']'):
        try:
            return ast.literal_eval(val_str)
        except:
            pass
    # Fallback: treat as a standard comma-separated string
    return [s.strip() for s in val_str.split(',') if s.strip()]

# Parse the services column for the entire dataframe
df['parsed_services'] = df['services'].apply(parse_services)

num_iterations = 5000
np.random.seed(42)  # Ensures reproducible results across runs

# 2. Iterate through each architectural group to execute the bootstrap simulation
for group_name, col_name in groups.items():
    if col_name not in df.columns:
        print(f"Warning: Column '{col_name}' not found in the dataset. Skipping group '{group_name}'.")
        continue

    # Filter rows where the architecture belongs to the current group
    # cast to bool handles both numeric (1/0) and boolean (True/False) configurations
    group_df = df[df[col_name].astype(bool)].copy()
    group_size = len(group_df)

    if group_size == 0:
        print(f"\nGroup '{group_name}' contains 0 architectures in this file. Skipping.")
        continue

    # Extract and aggregate all services used in this specific group
    all_services = []
    for s_list in group_df['parsed_services']:
        all_services.extend(s_list)

    # Count original frequencies and identify the top 15 most popular services
    service_counts = Counter(all_services)
    target_services = [item[0] for item in service_counts.most_common(15)]

    # Generate the binary matrix (0 or 1) representing service selection per architecture
    matrix_data = []
    for s_list in group_df['parsed_services']:
        matrix_data.append({service: int(service in s_list) for service in target_services})
    df_matrix = pd.DataFrame(matrix_data)

    # Matrix to store recorded ranks across all 5000 iterations
    # Shape: (5000 iterations, number of target services)
    rank_history = np.zeros((num_iterations, len(target_services)))

    for i in range(num_iterations):
        # Resample the group's architectures with replacement (keeping size exactly equal to group_size)
        bootstrap_sample = df_matrix.sample(n=group_size, replace=True)
        simulated_counts = bootstrap_sample.sum()

        # Rank services descending (higher count = lower ordinal rank number e.g., 1st, 2nd)
        # method='first' handles ties consistently based on dataframe index order
        ranks = (-simulated_counts).rank(method='first').values
        rank_history[i, :] = ranks

    # 3. Process and compile summary results for this group
    summary_data = []
    for idx, service in enumerate(target_services):
        service_ranks = rank_history[:, idx]

        mean_rank = np.mean(service_ranks)
        ci_lower = np.percentile(service_ranks, 2.5)
        ci_upper = np.percentile(service_ranks, 97.5)
        top_2_pct = np.mean((service_ranks <= 2)) * 100

        summary_data.append({
            'Service': service,
            'Original Count': service_counts[service],
            'Avg Rank Position': round(mean_rank, 1),
            '95% Rank CI': f"[{int(ci_lower)} to {int(ci_upper)}]",
            'Top 2 Finish Frequency': f"{top_2_pct:.1f}%"
        })

    # Display results formatted clearly for your paper
    df_results = pd.DataFrame(summary_data).sort_values(by='Avg Rank Position')
    print(f"\n" + "="*75)
    print(f"--- BOOTSTRAP RANK STABILITY ANALYSIS: {group_name.upper()} (N={group_size}) ---")
    print(f"="*75)
    print(df_results.to_string(index=False))


--- BOOTSTRAP RANK STABILITY ANALYSIS: HPC (N=15) ---
              Service  Original Count  Avg Rank Position 95% Rank CI Top 2 Finish Frequency
                  EC2              11                1.6    [1 to 4]                  87.3%
                   S3              11                1.8    [1 to 4]                  83.0%
               Lambda               8                3.8    [1 to 7]                  15.0%
           ThirdParty               8                4.0    [2 to 8]                  11.2%
                  FSX               6                6.0   [2 to 12]                   2.8%
                  EKS               5                7.4   [3 to 14]                   0.3%
           CloudFront               4                8.9   [4 to 14]                   0.1%
                Batch               4                9.2   [5 to 15]                   0.0%
   UserCompanyAnalyst               4                9.2   [4 to 14]                   0.1%
      UserConsumerWeb    

# Top Functional Goals, Per Group

In [ ]:
import pandas as pd
import numpy as np
import ast
from collections import Counter

# 1. Specify your dataset filename here
FILE_PATH = 'classified_architectures_with_metadata.csv'

try:
    df = pd.read_csv(FILE_PATH)
except FileNotFoundError:
    print(f"Error: File '{FILE_PATH}' not found. Please update the FILE_PATH variable with your actual filename.")
    exit()

# Define the groups and their corresponding binary indicator columns from your file
groups = {
    'hpc': 'belongs_hpc',
    'edge': 'belongs_edge',
    'serverless': 'belongs_serverless',
    'faas': 'belongs_serverless_faas',
    'strictly_edge': 'belongs_edge_no_cloudfront',
    'none': 'belongs_none_analytical'
}

# Robust helper function to parse the 'category' column
def parse_category(val):
    if pd.isna(val):
        return []
    val_str = str(val).strip()
    # Check if it is formatted as a Python list string e.g., ['S3', 'EC2']
    if val_str.startswith('[') and val_str.endswith(']'):
        try:
            return ast.literal_eval(val_str)
        except:
            pass
    # Fallback: treat as a standard comma-separated string
    return [s.strip() for s in val_str.split(',') if s.strip()]

# Parse the category column for the entire dataframe
df['parsed_category'] = df['category'].apply(parse_category)

num_iterations = 5000
np.random.seed(42)  # Ensures reproducible results across runs

# 2. Iterate through each architectural group to execute the bootstrap simulation
for group_name, col_name in groups.items():
    if col_name not in df.columns:
        print(f"Warning: Column '{col_name}' not found in the dataset. Skipping group '{group_name}'.")
        continue

    # Filter rows where the architecture belongs to the current group
    # cast to bool handles both numeric (1/0) and boolean (True/False) configurations
    group_df = df[df[col_name].astype(bool)].copy()
    group_size = len(group_df)

    if group_size == 0:
        print(f"\nGroup '{group_name}' contains 0 architectures in this file. Skipping.")
        continue

    # Extract and aggregate all category used in this specific group
    all_category = []
    for s_list in group_df['parsed_category']:
        all_category.extend(s_list)

    # Count original frequencies and identify the top 15 most popular category
    category_counts = Counter(all_category)
    target_category = [item[0] for item in category_counts.most_common(15)]

    # Generate the binary matrix (0 or 1) representing category selection per architecture
    matrix_data = []
    for s_list in group_df['parsed_category']:
        matrix_data.append({category: int(category in s_list) for category in target_category})
    df_matrix = pd.DataFrame(matrix_data)

    # Matrix to store recorded ranks across all 5000 iterations
    # Shape: (5000 iterations, number of target category)
    rank_history = np.zeros((num_iterations, len(target_category)))

    for i in range(num_iterations):
        # Resample the group's architectures with replacement (keeping size exactly equal to group_size)
        bootstrap_sample = df_matrix.sample(n=group_size, replace=True)
        simulated_counts = bootstrap_sample.sum()

        # Rank category descending (higher count = lower ordinal rank number e.g., 1st, 2nd)
        # method='first' handles ties consistently based on dataframe index order
        ranks = (-simulated_counts).rank(method='first').values
        rank_history[i, :] = ranks

    # 3. Process and compile summary results for this group
    summary_data = []
    for idx, category in enumerate(target_category):
        category_ranks = rank_history[:, idx]

        mean_rank = np.mean(category_ranks)
        ci_lower = np.percentile(category_ranks, 2.5)
        ci_upper = np.percentile(category_ranks, 97.5)
        top_2_pct = np.mean((category_ranks <= 2)) * 100

        summary_data.append({
            'Category': category,
            'Original Count': category_counts[category],
            'Avg Rank Position': round(mean_rank, 1),
            '95% Rank CI': f"[{int(ci_lower)} to {int(ci_upper)}]",
            'Top 2 Finish Frequency': f"{top_2_pct:.1f}%"
        })

    # Display results formatted clearly for your paper
    df_results = pd.DataFrame(summary_data).sort_values(by='Avg Rank Position')
    print(f"\n" + "="*75)
    print(f"--- BOOTSTRAP RANK STABILITY ANALYSIS: {group_name.upper()} (N={group_size}) ---")
    print(f"="*75)
    print(df_results.to_string(index=False))


--- BOOTSTRAP RANK STABILITY ANALYSIS: HPC (N=15) ---
         Category  Original Count  Avg Rank Position 95% Rank CI Top 2 Finish Frequency
compute_intensive               8                1.3    [1 to 3]                  96.5%
      interactive               5                2.4    [1 to 3]                  56.5%
   data_ingestion               5                2.4    [1 to 3]                  46.6%
          control               1                4.0    [3 to 4]                   0.4%

--- BOOTSTRAP RANK STABILITY ANALYSIS: EDGE (N=106) ---
         Category  Original Count  Avg Rank Position 95% Rank CI Top 2 Finish Frequency
   data_ingestion              49                1.4    [1 to 2]                 100.0%
      interactive              47                1.6    [1 to 2]                 100.0%
compute_intensive              18                3.0    [3 to 3]                   0.0%
          control               8                4.1    [4 to 5]                   0.0%
        